# HPD 4 — Entregable: Servidor MCP y conexión segura a datos

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd4-entregable.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 %
>
> **Plazo:** 7 días tras la sesión presencial.
>
> **Prerrequisito:** [HPD 4 — Servidor
> MCP](../bloque2/hpd4-servidor-mcp.qmd).
>
> **Entrega:** Notebook `.ipynb` ejecutado con todas las celdas
> completas. Cada ejercicio especifica qué variable debe contener el
> resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto (`eval1_server`, `eval2_respuesta`, `eval3_ataques`,
> `eval4_riesgos`). El script de corrección ejecutará tu notebook e
> inspeccionará esas variables.

## Setup

In [1]:
!pip install -q fastmcp==3.3.1 openai==2.38.0 python-dotenv==1.2.2 "pandas<3"

In [2]:
import os, sqlite3, tempfile, json, re, time, urllib.request
from pathlib import Path
"""
Setup compartido para la API key de Llamus en todos los notebooks.
Inyectado en cada .qmd/.ipynb vía scripts/sync_setup_keys.py (pre-render).

Orden de resolución:
  1. .env local (python-dotenv)
  2. os.environ ya configurado externamente
  3. Google Colab userdata: secret "LLAMUS_API_KEY" (preferente) o "LLM_API_KEY"

Las celdas posteriores pueden leer la clave con os.environ.get("LLM_API_KEY").
"""

import os

def _load_llm_api_key():
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    if not os.getenv("LLM_API_KEY"):
        try:
            from google.colab import userdata
            for name in ("LLAMUS_API_KEY", "LLM_API_KEY"):
                try:
                    value = userdata.get(name)
                    if value:
                        os.environ["LLM_API_KEY"] = value
                        break
                except Exception:
                    continue
        except ImportError:
            pass

_load_llm_api_key()

if os.getenv("LLM_API_KEY"):
    print("✓ LLM_API_KEY cargada.")
else:
    print("⚠️  LLM_API_KEY no encontrada.")
    print("    • Local: crea un archivo .env con LLM_API_KEY=tu_clave")
    print("    • Colab: añade un Secret 🔑 con nombre 'LLAMUS_API_KEY'")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
import warnings
warnings.filterwarnings("ignore")

import numpy as np
np.random.seed(42)
import pandas as pd

print("Librerías cargadas correctamente.")

✓ LLM_API_KEY cargada.
Librerías cargadas correctamente.

In [3]:
LLM_KEY = os.getenv("LLM_API_KEY")
LLM_URL = os.getenv("LLM_URL", "https://llamus.cs.us.es/api/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "gemma4:26b")

if LLM_KEY:
    from openai import OpenAI
    client = OpenAI(base_url=LLM_URL, api_key=LLM_KEY)
    print("✅ Cliente OpenAI configurado.")
else:
    client = None
    print("⚠️  LLM_API_KEY no encontrada. El ejercicio E2 requiere la clave.")

✅ Cliente OpenAI configurado.

In [4]:
# Función auxiliar compartida: ejecutar herramienta del servidor MCP (simulación en notebook)
from fastmcp import FastMCP

TOOLS_MCP = {}

def ejecutar_tool(nombre: str, args: dict) -> str:
    if nombre not in TOOLS_MCP:
        return f"Error: tool '{nombre}' no encontrada. Disponibles: {list(TOOLS_MCP.keys())}"
    try:
        return TOOLS_MCP[nombre](**args)
    except Exception as e:
        return f"Error ejecutando {nombre}: {e}"

print("✅ Helper de ejecución de tools listo.")

✅ Helper de ejecución de tools listo.

In [5]:
CHINOOK_URL = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"
CHINOOK_PATH = os.path.join(tempfile.gettempdir(), "chinook_cache", "Chinook_Sqlite.sqlite")

def _es_chinook(path) -> bool:
    """Comprueba que el fichero es SQLite y contiene la tabla Invoice."""
    try:
        con = sqlite3.connect(f"{Path(path).resolve().as_uri()}?mode=ro", uri=True)
        try:
            return con.execute(
                "SELECT 1 FROM sqlite_master WHERE type='table' AND name='Invoice'"
            ).fetchone() is not None
        finally:
            con.close()
    except sqlite3.DatabaseError:
        return False

def descargar_chinook(url=CHINOOK_URL, destino=CHINOOK_PATH, timeout=60) -> str:
    """Descarga Chinook (~1 MB) al directorio temporal. Si ya existe la copia en caché, no la descarga de nuevo."""
    if os.path.exists(destino) and _es_chinook(destino):
        return destino
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    tmp = destino + ".part"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            data = resp.read()
        with open(tmp, "wb") as f:
            f.write(data)
    except Exception as e:
        raise RuntimeError(
            f"No se pudo descargar la base Chinook desde:\n  {url}\n"
            f"Causa: {e}\n"
            "Comprueba la conexión a Internet. Si el problema persiste, descarga el fichero "
            f"manualmente y guárdalo en:\n  {destino}"
        ) from e
    if not _es_chinook(tmp):
        os.remove(tmp)
        raise RuntimeError("El fichero descargado no es la base Chinook (SQLite con tabla Invoice).")
    os.replace(tmp, destino)
    return destino

def conectar_solo_lectura(path: str):
    """Abre la base en modo solo lectura (mode=ro) y con un autorizador que solo permite leer:
    el motor rechaza ATTACH, PRAGMA, escrituras y cualquier otra acción aunque el validador falle."""
    uri = Path(path).resolve().as_uri() + "?mode=ro"
    con = sqlite3.connect(uri, uri=True, check_same_thread=False)
    permitidas = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ, sqlite3.SQLITE_FUNCTION}
    con.set_authorizer(lambda accion, *_: sqlite3.SQLITE_OK if accion in permitidas else sqlite3.SQLITE_DENY)
    return con

DB_PATH = descargar_chinook()
db_e = conectar_solo_lectura(DB_PATH)
n_tablas = db_e.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='table'").fetchone()[0]
print(f"✅ Chinook lista en modo solo lectura: {n_tablas} tablas ({DB_PATH})")

✅ Chinook lista en modo solo lectura: 11 tablas (/var/folders/5q/dq_s1n552d37rqhcc02ppsn00000gn/T/chinook_cache/Chinook_Sqlite.sqlite)

## Contexto: facturación de una tienda de música digital

**Contexto:** Trabajas como analista financiero en una tienda de música
digital. Vendes canciones y álbumes online y toda la facturación queda
registrada en una base de datos (Chinook). El equipo de finanzas quiere
que los analistas consulten ventas, clientes, países y géneros mediante
un asistente conversacional. Para ello construyes un servidor MCP de
**solo lectura**: el asistente puede consultar la facturación, pero
nunca modificarla.

### Base de datos — Chinook

Las tablas principales son: `Invoice` (facturas: fecha, país de
facturación, total), `InvoiceLine` (líneas de factura), `Customer`
(clientes), `Track`, `Album`, `Artist`, `Genre` (catálogo) y `Employee`
(empleados, con datos personales). Ejecuta `print(DB_PATH)` o consulta
`sqlite_master` para explorar el esquema completo.

> **Procedencia de los datos**
>
> La base es **Chinook** (Luis Rocha, © 2008–2024), publicada bajo
> **licencia MIT** en el repositorio
> [lerocha/chinook-database](https://github.com/lerocha/chinook-database).
> Según su autor, los datos de catálogo proceden de una biblioteca de
> iTunes real, mientras que clientes y empleados son ficticios y las
> ventas se generaron de forma aleatoria. El fichero SQLite se descarga
> la primera vez y queda guardado en caché en el directorio temporal.

### Ejercicio 1 — Servidor MCP de solo lectura (3 ptos)

Construye un servidor FastMCP que exponga las siguientes herramientas
sobre la base Chinook:

- `listar_tablas()` — devuelve la lista de tablas disponibles
- `describir_tabla(tabla: str)` — devuelve el DDL de la tabla
- `consultar_sql(query: str)` — ejecuta una consulta SELECT y devuelve
  resultados
- Resource `resumen://facturacion` — devuelve un resumen de la
  facturación (total, número de facturas y clientes, periodo y top 5
  países)

Requisitos de seguridad de `consultar_sql` (los verificarás en el
ejercicio 3):

- Solo se admiten consultas `SELECT`, y una única sentencia por
  consulta.
- Se rechazan palabras de escritura o administración (`DROP`, `UPDATE`,
  `DELETE`, `INSERT`, `ALTER`, `CREATE`, `TRUNCATE`, `ATTACH`, `DETACH`,
  `PRAGMA`, `VACUUM`).
- Devuelve como máximo 30 filas y 2000 caracteres.
- Cancela la consulta si tarda más de 5 segundos.
- Los errores se devuelven como texto, nunca como excepción.

La conexión `db_e` ya está abierta en modo solo lectura (`mode=ro`) en
la celda de setup.

Asigna el servidor a `eval1_server` (como string con los logs de
creación y verificación).

In [6]:
mcp_e1 = FastMCP("Servidor Facturación Chinook (solo lectura)")

@mcp_e1.tool()
def listar_tablas() -> str:
    """Devuelve la lista de tablas disponibles en la base de datos de facturación."""
    # TODO: consultar sqlite_master (type='table') y devolver los nombres de las tablas
    return "⚠️ TODO: implementar listar_tablas"

@mcp_e1.tool()
def describir_tabla(tabla: str) -> str:
    """Devuelve el DDL (CREATE TABLE) de una tabla de la base de datos.

    Args:
        tabla: nombre de la tabla (por ejemplo 'Invoice').
    """
    # TODO: devolver el DDL de la tabla. Usa parámetros (?) en la consulta, no concatenes el texto del usuario.
    # TODO: si la tabla no existe, devolver un mensaje de error (no lanzar excepción)
    return "⚠️ TODO: implementar describir_tabla"

@mcp_e1.tool()
def consultar_sql(query: str) -> str:
    """Ejecuta una consulta SQL SELECT de solo lectura sobre la base de facturación.

    Args:
        query: consulta SQL (solo SELECT, una sentencia). Máximo 30 filas.
    """
    # TODO: validar la consulta (SELECT únicamente, una sola sentencia, sin palabras prohibidas)
    # TODO: ejecutar con db_e, limitar a 30 filas y 2000 caracteres
    # TODO: timeout de 5 s (pista: sqlite3 tiene Connection.set_progress_handler)
    # TODO: capturar errores y devolverlos como texto que empiece por "Error"
    return "⚠️ TODO: implementar consultar_sql"

@mcp_e1.resource("resumen://facturacion")
def resumen_facturacion() -> str:
    """Resumen agregado de la facturación: total, número de facturas y clientes, periodo y top 5 países."""
    # TODO: calcular total facturado, número de facturas, clientes con compras, periodo (MIN/MAX de InvoiceDate)
    # TODO: añadir el top 5 de países por facturación (tabla Invoice, columna BillingCountry)
    return "⚠️ TODO: implementar resumen_facturacion"

TOOLS_MCP.clear()
TOOLS_MCP.update({
    "listar_tablas": listar_tablas,
    "describir_tabla": describir_tabla,
    "consultar_sql": consultar_sql,
})

eval1_server = "✅ Servidor MCP de facturación creado con 3 tools + 1 resource\n"
eval1_server += "  Tools: listar_tablas, describir_tabla, consultar_sql\n"
eval1_server += "  Resource: resumen://facturacion\n"
eval1_server += "\n--- Prueba: listar_tablas ---\n"
eval1_server += listar_tablas()
eval1_server += "\n\n--- Prueba: resource resumen://facturacion ---\n"
eval1_server += resumen_facturacion()
print(eval1_server)

✅ Servidor MCP de facturación creado con 3 tools + 1 resource
  Tools: listar_tablas, describir_tabla, consultar_sql
  Resource: resumen://facturacion

--- Prueba: listar_tablas ---
⚠️ TODO: implementar listar_tablas

--- Prueba: resource resumen://facturacion ---
⚠️ TODO: implementar resumen_facturacion

### Ejercicio 2 — Consulta en lenguaje natural (2 ptos)

Conecta el cliente OpenAI al servidor MCP. Formula una pregunta de
negocio en lenguaje natural (sobre facturación, clientes, países o
géneros), deja que el LLM decida qué tool usar y muestra la respuesta
final.

> **El modelo no sabe en qué fecha estamos**
>
> Si tu pregunta se refiere a un periodo, **escríbelo explícitamente**
> («en 2011», «entre 2010 y 2012»…) usando el rango de fechas que
> calculaste en el ejercicio 1. Con expresiones como «este año» o «el
> último trimestre», el LLM se inventa un año y la consulta puede
> devolver 0 resultados.

Asigna el resultado (pregunta + respuesta completa) a `eval2_respuesta`.

In [7]:
# TODO: define las tres tools en formato OpenAI function calling (listar_tablas, describir_tabla, consultar_sql).
#   Para consultar_sql, describe en "description" las tablas y columnas principales (ayuda al LLM a escribir SQL).
tools_definition_e2 = []

def preguntar_al_llm(pregunta, tools_def, system_prompt=None, max_rondas=4):
    """
    Envía la pregunta al LLM. Mientras el LLM pida tools (tool_calls), ejecútalas con
    ejecutar_tool() y devuelve los resultados al historial; cuando no pida más, devuelve su respuesta.
    """
    if client is None:
        return "⚠️  Cliente OpenAI no disponible."
    # TODO: montar messages (system + user) y bucle con client.chat.completions.create(
    #       model=LLM_MODEL, messages=..., tools=tools_def, tool_choice="auto", max_tokens=2048)
    # TODO: si message.tool_calls, añadir el mensaje del asistente y un mensaje role="tool" por cada tool_call
    # TODO: si no hay tool_calls, devolver message.content
    return "⚠️ TODO: implementar preguntar_al_llm"

# TODO: escribe aquí tu pregunta de negocio (cadena de texto)
pregunta_e2 = None

if client is None:
    eval2_respuesta = "⚠️  Cliente OpenAI no disponible. Falta LLM_API_KEY."
elif pregunta_e2 is None:
    eval2_respuesta = "⚠️  TODO: define pregunta_e2 y ejecuta el bucle de tool calling."
else:
    respuesta_e2 = preguntar_al_llm(pregunta_e2, tools_definition_e2)
    eval2_respuesta = f"PREGUNTA: {pregunta_e2}\n\nRESPUESTA: {respuesta_e2}"

print(eval2_respuesta)

⚠️  TODO: define pregunta_e2 y ejecuta el bucle de tool calling.

### Ejercicio 3 — Pruebas de inyección y límites (1.5 ptos)

Prueba al menos 3 intentos de ataque (inyección SQL o manipulación del
prompt) contra tu servidor MCP, y comprueba también el límite de filas y
el timeout. Documenta qué intentaste, qué respuesta obtuvo y si la
defensa funcionó.

Cada caso indica en `esperado` una parte del mensaje que devuelve el
servidor cuando la defensa funciona (por ejemplo `"Error"`,
`"más filas"` o `"timeout"`).

Asigna los resultados a `eval3_ataques` (lista de diccionarios con
ataque, resultado, defendido).

In [8]:
ataques_e3 = [
    {
        "ataque": "DROP TABLE Customer",
        "tipo": "SQL directo (destructivo)",
        "query": "DROP TABLE Customer",
        "esperado": "Error",
    },
    # TODO: añade al menos 2 ataques más (mismo formato), por ejemplo:
    #   - una sentencia UPDATE o DELETE sobre Invoice (manipular la facturación)
    #   - varias sentencias separadas por ';' con una sentencia de escritura
    #   - PRAGMA o ATTACH (administración de la base)
    #   - una consulta sin límite de filas (p. ej. sobre Track) -> esperado: "más filas"
    #   - una consulta muy costosa (producto cartesiano de varias tablas) -> esperado: "timeout"
]

eval3_ataques = []
for ataque in ataques_e3:
    resultado = consultar_sql(ataque["query"])
    defendido = ataque["esperado"] in resultado
    eval3_ataques.append({
        "ataque": ataque["ataque"],
        "tipo": ataque["tipo"],
        "resultado": resultado[:200],
        "defendido": defendido,
        "defensa_efectiva": "✅ Defendido" if defendido else "❌ No defendido",
    })

for e in eval3_ataques:
    print(f"\n--- {e['tipo']} ---")
    print(f"  Ataque: {e['ataque']}")
    print(f"  Resultado: {e['defensa_efectiva']}")
    print(f"  Detalle: {e['resultado']}")


--- SQL directo (destructivo) ---
  Ataque: DROP TABLE Customer
  Resultado: ❌ No defendido
  Detalle: ⚠️ TODO: implementar consultar_sql

### Ejercicio 4 — Análisis de riesgos (1.5 ptos)

Redacta un análisis de riesgos para el servidor MCP en un entorno
financiero real. Debes cubrir al menos:

- Riesgos de privacidad (datos personales y de contacto de clientes y
  empleados). **Demuéstralo:** usa tu servidor para obtener datos
  personales de `Employee` (nombre y correo, por ejemplo) y pega en el
  análisis la consulta que has usado y una muestra del resultado. Tu
  servidor permite cualquier `SELECT`, así que debes comprobar qué se
  filtra y proponer cómo evitarlo.
- Inyección SQL y prompt injection
- Data leakage (fuga de información por respuestas demasiado detalladas)
- Mitigaciones específicas para cada riesgo

Puedes apoyarte en estas preguntas (no es necesario responderlas todas):

- ¿Qué columnas de `Customer` y `Employee` son datos personales?
  ¿Deberían llegar al asistente?
- ¿Qué pasa si un campo de texto de la base (p. ej. un nombre de
  canción) contiene instrucciones dirigidas al LLM?
- ¿Qué protege la conexión `mode=ro` que no protege la validación de la
  consulta?
- ¿Qué consultas podrían bloquear el servidor y cómo lo evitarías?

Asigna tu respuesta a `eval4_riesgos`.

In [9]:
eval4_riesgos = """TODO: escribe aquí el análisis de riesgos, con estas secciones:

RIESGOS DE PRIVACIDAD:
- TODO

INYECCIÓN SQL / PROMPT INJECTION:
- TODO

DATA LEAKAGE:
- TODO

RIESGOS ADICIONALES:
- TODO
"""
print("✅ eval4_riesgos asignada.")

✅ eval4_riesgos asignada.

## Resumen de entregables

In [10]:
print(f"\nVariables de corrección:")
for var in ["eval1_server", "eval2_respuesta", "eval3_ataques", "eval4_riesgos"]:
    val = globals().get(var, None)
    status = "✅ definida" if val is not None else "⚠️ no definida"
    print(f"  {var}: {status}")


Variables de corrección:
  eval1_server: ✅ definida
  eval2_respuesta: ✅ definida
  eval3_ataques: ✅ definida
  eval4_riesgos: ✅ definida

> **Entregable**
>
> - Servidor MCP funcional con 3 tools (listar_tablas, describir_tabla,
>   consultar_sql) + 1 resource específico del dominio
> - Consulta en lenguaje natural ejecutada con respuesta del LLM
> - 3 pruebas de inyección documentadas con resultados
> - Análisis de riesgos con mitigaciones concretas
>
> ## Criterios de evaluación
>
> - **E1 (3 ptos):** Servidor MCP operativo con todas las tools y
>   resource implementados
> - **E2 (2 ptos):** Cliente LLM conectado correctamente y respuesta
>   coherente a la pregunta
> - **E3 (1.5 ptos):** Pruebas de seguridad con defensas efectivas
>   documentadas
> - **E4 (1.5 ptos):** Análisis de riesgos completo con mitigaciones
>   específicas del dominio
> - **Total: 7.5 % de la nota final**

# References